# Clean viruses → interim

Normalize multi-source virus detections onto modelling keys (spec `04`).

| Item | Detail |
|------|--------|
| Input | `data/01_raw/virus/virus_detections.tsv` + `countries.csv`, `subnational_locations.csv`, `host_groups.csv`, `vector_taxon_groups.csv` |
| Output | `data/02_interim/viruses_normalized.parquet` (incl. `host_group`); unmatched regions → `virus_unmatched_regions.csv`; unmatched hosts → `virus_unmatched_hosts.csv` |
| H3 | `H3_RESOLUTION` from `vector_guard.resolve` (locked = 4) |
| Time | `collection_date` → year-month (month required for panel keys) |
| Coords | Locked ladder: exact GPS / backfill → curated centroid → `country_only` |
| Host | Free-text `host` (+ raw `isolation_source` fallback) → closed `host_group`; drop `isolation_source` from interim |

**Guardrail:** never call external geocoders. Unmatched region strings stay `country_only` and are listed for optional `05` follow-up. Unmatched hosts stay `host_group=unknown` and are listed for optional `06b` follow-up.

In [1]:
from __future__ import annotations

from pathlib import Path

import pandas as pd
from IPython.display import display

from vector_guard.data import (
    default_interim_dir,
    find_repo_root,
    load_virus_detections,
)
from vector_guard.resolve import (
    GEO_PRECISION_CENTROID,
    GEO_PRECISION_COUNTRY,
    GEO_PRECISION_EXACT,
    GEO_PRECISIONS,
    H3_RESOLUTION,
    HOST_GROUPS,
    assign_h3,
    build_host_alias_map,
    build_host_taxon_lookup,
    build_name_to_iso2,
    build_subnational_centroid_index,
    classify_host_group,
    load_countries,
    load_host_groups,
    load_subnational_locations,
    load_vector_taxon_groups,
    lookup_subnational_centroid,
    norm_key,
    parse_year_month_virus,
    region_candidate_keys,
    resolve_coords_ladder,
    resolve_virus_country_iso2,
)

REPO_ROOT = find_repo_root(Path.cwd())
INTERIM_DIR = default_interim_dir(REPO_ROOT)
INTERIM_DIR.mkdir(parents=True, exist_ok=True)
OUT_PATH = INTERIM_DIR / "viruses_normalized.parquet"
UNMATCHED_PATH = INTERIM_DIR / "virus_unmatched_regions.csv"
UNMATCHED_HOSTS_PATH = INTERIM_DIR / "virus_unmatched_hosts.csv"

assert H3_RESOLUTION is not None and H3_RESOLUTION >= 0
REPO_ROOT, OUT_PATH, UNMATCHED_PATH, UNMATCHED_HOSTS_PATH, H3_RESOLUTION

(PosixPath('/workspaces/vector-guard'),
 PosixPath('/workspaces/vector-guard/data/02_interim/viruses_normalized.parquet'),
 PosixPath('/workspaces/vector-guard/data/02_interim/virus_unmatched_regions.csv'),
 PosixPath('/workspaces/vector-guard/data/02_interim/virus_unmatched_hosts.csv'),
 4)

## Load raw + reference

In [2]:
raw = load_virus_detections()
countries = load_countries()
subnational = load_subnational_locations()
host_groups_ref = load_host_groups()
vector_taxa = load_vector_taxon_groups()

name_to_iso2 = build_name_to_iso2(countries)
centroid_index = build_subnational_centroid_index(subnational)
host_alias_map = build_host_alias_map(host_groups_ref)
host_taxon_lookup = build_host_taxon_lookup(vector_taxa)

{
    "raw_rows": len(raw),
    "countries": len(countries),
    "subnational": len(subnational),
    "host_group_aliases": len(host_alias_map),
    "host_taxon_keys": len(host_taxon_lookup),
    "sources": sorted(raw["source"].fillna("").unique().tolist()),
}

{'raw_rows': 168307,
 'countries': 251,
 'subnational': 7798,
 'host_group_aliases': 403,
 'host_taxon_keys': 251,
 'sources': ['bvbrc',
  'ecdc_atlas',
  'empres_i',
  'ena_read_run',
  'ena_sequence',
  'pathoplexus']}

## Resolve ISO-2, year-month, coords, H3

Coord priority (locked):

1. Valid `lat`/`lon` when `coord_source` ≠ `none` → `exact_coords`
2. `(country_iso2, subnational_region | nuts_id)` → `subnational_locations.csv` → `subnational_centroid`
3. Else → `country_only`, `h3_cell` null

In [3]:
def _text(value: object) -> str:
    text = str(value or "").strip()
    if text.lower() in {"nan", "none", "null", "na", "n/a", "<na>", "no_value"}:
        return ""
    return text


records = raw.to_dict(orient="records")

country_iso2 = [
    resolve_virus_country_iso2(row.get("country", ""), name_to_iso2) for row in records
]
subnational_region = [_text(row.get("subnational_region", "")) for row in records]
nuts_id = [_text(row.get("nuts_id", "")) for row in records]
coord_source = [_text(row.get("coord_source", "")).lower() for row in records]

year_month = [parse_year_month_virus(row.get("collection_date", "")) for row in records]
year = [pair[0] if pair else pd.NA for pair in year_month]
month = [pair[1] if pair else pd.NA for pair in year_month]

coords = [
    resolve_coords_ladder(
        lat=row.get("lat", ""),
        lon=row.get("lon", ""),
        coord_source=src,
        country_iso2=iso2,
        subnational_region=region,
        nuts_id=nuts,
        centroid_index=centroid_index,
        allow_centroid=True,
    )
    for row, src, iso2, region, nuts in zip(
        records, coord_source, country_iso2, subnational_region, nuts_id, strict=True
    )
]
lat = [item[0] for item in coords]
lon = [item[1] for item in coords]
geo_precision = [item[2] for item in coords]
h3_cell = [assign_h3(la, lo, H3_RESOLUTION) for la, lo in zip(lat, lon, strict=True)]

host = [_text(row.get("host", "")) for row in records]
isolation_source = [_text(row.get("isolation_source", "")) for row in records]

# Classify unique (host, isolation_source) pairs once, then map (spec 04).
pair_keys = list(dict.fromkeys(zip(host, isolation_source, strict=True)))
host_group_lookup = {
    (h, iso): classify_host_group(
        h,
        host_alias_map=host_alias_map,
        taxon_to_group=host_taxon_lookup,
        isolation_source=iso,
    )
    for h, iso in pair_keys
}
host_group = [host_group_lookup[(h, iso)] for h, iso in zip(host, isolation_source, strict=True)]

out = pd.DataFrame(
    {
        "record_id": [_text(row.get("record_id", "")) for row in records],
        "source": [_text(row.get("source", "")) for row in records],
        "pathogen_name": [_text(row.get("pathogen_name", "")) for row in records],
        "vector_group": [_text(row.get("vector_group", "")) for row in records],
        "country_iso2": country_iso2,
        "subnational_region": subnational_region,
        "nuts_id": nuts_id,
        "lat": lat,
        "lon": lon,
        "coord_source": coord_source,
        "geo_precision": geo_precision,
        "h3_cell": h3_cell,
        "year": year,
        "month": month,
        "accession": [_text(row.get("accession", "")) for row in records],
        "host": host,
        "host_group": host_group,
    }
)
# Keep isolation_source only for unmatched-host audit below; not an interim column.
out_isolation_source = pd.Series(isolation_source, dtype="string")

out["year"] = pd.to_numeric(out["year"], errors="coerce").astype("Int64")
out["month"] = pd.to_numeric(out["month"], errors="coerce").astype("Int64")

print(
    f"host_group pairs classified: {len(host_group_lookup):,} unique "
    f"→ {len(out):,} rows"
)
display(out["host_group"].value_counts())
out.head()

host_group pairs classified: 4,314 unique → 168,307 rows


host_group
human              84642
livestock          40653
unknown            14587
mosquito           11815
tick                6538
bird                5060
wildlife_mammal     2962
lab                 1301
biting_midge         488
sandfly              261
Name: count, dtype: int64

,record_id,source,pathogen_name,vector_group,country_iso2,subnational_region,nuts_id,lat,lon,coord_source,geo_precision,h3_cell,year,month,accession,host,host_group
0,ena_sequence:MH170264,ena_sequence,West Nile virus,mosquito,US,"GA, Muscogee County",,32.8547,-83.4078,,subnational_centroid,8444c01ffffffff,<NA>,<NA>,MH170264,Baeolophus bicolor,bird
1,ena_sequence:MH819470,ena_sequence,West Nile virus,mosquito,CA,,,NaN,NaN,,country_only,NaN,2014,9,MH819470,Culex sp.,mosquito
2,ena_sequence:MW803141,ena_sequence,West Nile virus,mosquito,RS,Zemun,,NaN,NaN,,country_only,NaN,<NA>,<NA>,MW803141,Culex pipiens,mosquito
3,ena_sequence:MW835355,ena_sequence,West Nile virus,mosquito,IT,,,NaN,NaN,,country_only,NaN,<NA>,<NA>,MW835355,magpie,bird
4,ena_sequence:MW835358,ena_sequence,West Nile virus,mosquito,IT,,,NaN,NaN,,country_only,NaN,<NA>,<NA>,MW835358,carrion crow,bird


## QA coverage

In [4]:
n = len(out)
tier1 = out["h3_cell"].notna()
qa = {
    "rows": n,
    "iso2_mapped_share": float((out["country_iso2"] != "").mean()),
    "year_month_share": float((out["year"].notna() & out["month"].notna()).mean()),
    "vector_group_share": float((out["vector_group"] != "").mean()),
    "tier1_h3_share": float(tier1.mean()),
    "tier2_country_only_share": float((out["geo_precision"] == GEO_PRECISION_COUNTRY).mean()),
    "exact_among_tier1": float(
        (out.loc[tier1, "geo_precision"] == GEO_PRECISION_EXACT).mean()
    )
    if tier1.any()
    else float("nan"),
    "centroid_among_tier1": float(
        (out.loc[tier1, "geo_precision"] == GEO_PRECISION_CENTROID).mean()
    )
    if tier1.any()
    else float("nan"),
}
display(pd.Series(qa))

print("geo_precision")
display(out["geo_precision"].value_counts(dropna=False))
print("coord_source")
display(out["coord_source"].value_counts(dropna=False))
print("vector_group")
display(out["vector_group"].value_counts(dropna=False))

print("host_group")
display(out["host_group"].value_counts(dropna=False))
host_non_empty = out["host"].ne("")
host_resolved = host_non_empty & out["host_group"].ne("unknown")
print(
    f"Non-empty host resolved share: "
    f"{float(host_resolved.sum() / host_non_empty.sum()) if host_non_empty.any() else float('nan'):.3%} "
    f"({int(host_resolved.sum()):,} / {int(host_non_empty.sum()):,})"
)
print("pathogen_name × host_group (top pathogens)")
top_pathogens = out["pathogen_name"].value_counts().head(8).index
display(
    pd.crosstab(
        out.loc[out["pathogen_name"].isin(top_pathogens), "pathogen_name"],
        out.loc[out["pathogen_name"].isin(top_pathogens), "host_group"],
    )
)

print("per-source geo")
display(
    out.groupby("source")
    .agg(
        rows=("record_id", "size"),
        iso2=("country_iso2", lambda s: (s != "").mean()),
        tier1=("h3_cell", lambda s: s.notna().mean()),
        ym=("year", lambda s: s.notna().mean()),
        host_unknown=("host_group", lambda s: (s == "unknown").mean()),
    )
    .sort_values("rows", ascending=False)
)

rows                        168307.000000
iso2_mapped_share                0.987553
year_month_share                 0.655166
vector_group_share               1.000000
tier1_h3_share                   0.460379
tier2_country_only_share         0.539621
exact_among_tier1                0.461380
centroid_among_tier1             0.538620
dtype: float64

geo_precision


geo_precision
country_only            90822
subnational_centroid    41735
exact_coords            35750
Name: count, dtype: int64

coord_source


coord_source
                   132557
exact_gps           34187
sample_backfill      1563
Name: count, dtype: int64

vector_group


... (3 outputs left out)

Non-empty host resolved share: 98.538% (151,489 / 153,736)
pathogen_name × host_group (top pathogens)


host_group,bird,biting_midge,human,lab,livestock,mosquito,tick,unknown,wildlife_mammal
pathogen_name,,,,,,,,,
Bluetongue virus,0,143,0,105,28166,0,0,3716,448
Chikungunya virus,0,0,11018,169,0,393,0,573,46
Crimean-Congo hemorrhagic fever virus,0,0,5762,6,94,0,2192,423,34
Dengue virus,0,1,60255,380,5,1462,0,4251,67
Japanese encephalitis virus,17,7,404,4,995,1912,32,375,22
Schmallenberg virus,0,77,6,0,3857,0,0,16,3
West Nile virus,3724,0,1570,48,1772,5887,18,1229,183
Zika virus,0,0,2740,343,54,607,0,684,294


per-source geo


,rows,iso2,tier1,ym,host_unknown
source,,,,,
ena_sequence,118404,0.993961,0.330698,0.569989,0.100478
empres_i,25874,0.998686,1.000000,1.0,0.014145
ena_read_run,13231,0.917013,0.539566,0.731388,0.133777
pathoplexus,6377,0.991846,0.564529,0.647954,0.037478
bvbrc,4108,0.952288,0.342989,0.677702,0.076680
ecdc_atlas,313,1.000000,0.980831,1.0,0.000000


## Unmatched subnational / NUTS keys (no geocoding)

Rows with region text or `nuts_id` that did **not** resolve to a centroid and have no exact GPS. Writes rebuildable `data/02_interim/virus_unmatched_regions.csv` with an `in_vectornet_scope` flag so notebook `05` curation can prioritize panel-relevant keys.

In [5]:
unmatched_mask = (
    (out["geo_precision"] == GEO_PRECISION_COUNTRY)
    & (out["country_iso2"] != "")
    & ((out["subnational_region"] != "") | (out["nuts_id"] != ""))
)

unmatched_rows = out.loc[unmatched_mask].copy()

vectornet_iso2 = {
    str(iso).strip().upper()
    for iso, flag in zip(
        countries["iso2"].fillna(""),
        countries["in_vectornet_scope"],
        strict=True,
    )
    if str(iso).strip()
    and str(flag).strip().lower() in {"1", "true", "yes"}
}


def region_lookup_key(row: pd.Series) -> str:
    """Display/report key aligned with runtime lookup candidates."""
    if row["nuts_id"]:
        return norm_key(row["nuts_id"])
    keys = region_candidate_keys(row["subnational_region"])
    return keys[0] if keys else ""


unmatched_rows["region_key"] = unmatched_rows.apply(region_lookup_key, axis=1)
# Confirm still unmatched under the same lookup used by the ladder
still_unmatched = []
for row in unmatched_rows.itertuples(index=False):
    hit = lookup_subnational_centroid(
        row.country_iso2,
        subnational_region=row.subnational_region,
        nuts_id=row.nuts_id,
        centroid_index=centroid_index,
    )
    if hit is None and row.region_key:
        still_unmatched.append(
            {
                "country_iso2": row.country_iso2,
                "region_key": row.region_key,
                "subnational_region": row.subnational_region,
                "nuts_id": row.nuts_id,
                "source": row.source,
            }
        )

unmatched_cols = [
    "country_iso2",
    "region_key",
    "subnational_region",
    "nuts_id",
    "source",
    "rows",
    "in_vectornet_scope",
]
unmatched = (
    pd.DataFrame(still_unmatched)
    .value_counts(
        ["country_iso2", "region_key", "subnational_region", "nuts_id", "source"]
    )
    .rename("rows")
    .reset_index()
    .sort_values("rows", ascending=False)
    if still_unmatched
    else pd.DataFrame(columns=unmatched_cols[:-1])
)
unmatched["in_vectornet_scope"] = unmatched["country_iso2"].isin(vectornet_iso2).astype(int)
unmatched = unmatched[unmatched_cols]

unmatched.to_csv(UNMATCHED_PATH, index=False)
vn_keys = int(unmatched["in_vectornet_scope"].sum())
vn_rows = int(unmatched.loc[unmatched["in_vectornet_scope"] == 1, "rows"].sum())
print(f"Wrote {UNMATCHED_PATH}")
print(
    f"Unmatched region/NUTS keys: {len(unmatched):,} "
    f"(VectorNet-scope keys={vn_keys:,}, rows={vn_rows:,})"
)
display(unmatched.head(40))
display(
    unmatched.loc[unmatched["in_vectornet_scope"] == 1]
    .head(20)
    .reset_index(drop=True)
)

Wrote /workspaces/vector-guard/data/02_interim/virus_unmatched_regions.csv
Unmatched region/NUTS keys: 1,875 (VectorNet-scope keys=628, rows=2,071)


,country_iso2,region_key,subnational_region,nuts_id,source,rows,in_vectornet_scope
0,VN,south,South,,ena_sequence,1216,0
1,CN,guangzhou,Guangzhou,,ena_sequence,674,0
2,VN,nha-trang,Nha Trang,,ena_sequence,562,0
3,CN,xi-shuang-ban-na,Xi Shuang Ban Na,,ena_sequence,259,0
4,VN,south,south,,ena_sequence,240,0
5,CN,hangzhou,Hangzhou,,ena_sequence,171,0
6,KE,mombasa,Mombasa,,ena_sequence,126,0
7,ID,surabaya,Surabaya,,ena_sequence,120,0
8,KH,kch,KCH,,ena_sequence,118,0
9,SA,jeddah,Jeddah,,ena_sequence,104,0


,country_iso2,region_key,subnational_region,nuts_id,source,rows,in_vectornet_scope
0,GR,rhodes,Rhodes,,ena_sequence,33,1
1,MA,azrou,Azrou,,ena_sequence,30,1
2,RS,tutin,Tutin,,ena_sequence,30,1
3,TN,ghardimaou,Ghardimaou,,ena_sequence,30,1
4,IL,jordan-valley,Jordan Valley,,ena_sequence,30,1
5,AT,vienna-international-airport,Vienna International Airport,,ena_sequence,22,1
6,XK,malisheve,Malisheve,,ena_sequence,21,1
7,IL,yokneam,Yokneam,,ena_sequence,21,1
8,PT,imported-unknown-origin,"imported, unknown origin",,ena_sequence,20,1
9,DZ,berrouaghia-media,Berrouaghia-Media,,ena_sequence,20,1


## Unmatched hosts (`host_group=unknown`)

Non-empty free-text `host` values that still resolve to `unknown` after alias + heuristic + `isolation_source` fallback. Written for optional `06b` curation — do **not** invent a group here.

In [6]:
_HOST_PLACEHOLDERS = {
    "missing",
    "unknown",
    "na",
    "n/a",
    "none",
    "null",
    "nan",
    "not applicable",
    "not collected",
}
unknown_host_mask = (
    out["host"].ne("")
    & ~out["host"].str.casefold().isin(_HOST_PLACEHOLDERS)
    & out["host_group"].eq("unknown")
)
unknown_hosts = out.loc[unknown_host_mask, ["host", "source"]].copy()
unknown_hosts["isolation_source"] = out_isolation_source.loc[unknown_host_mask].to_numpy()

if unknown_hosts.empty:
    unmatched_hosts = pd.DataFrame(
        columns=["host", "rows", "n_sources", "top_source", "sample_isolation_source"]
    )
else:
    unmatched_hosts = (
        unknown_hosts.groupby("host", dropna=False)
        .agg(
            rows=("source", "size"),
            n_sources=("source", "nunique"),
            top_source=("source", lambda s: s.value_counts().index[0]),
            sample_isolation_source=(
                "isolation_source",
                lambda s: next((x for x in s if str(x).strip()), ""),
            ),
        )
        .reset_index()
        .sort_values(["rows", "host"], ascending=[False, True])
        .reset_index(drop=True)
    )

unmatched_hosts.to_csv(UNMATCHED_HOSTS_PATH, index=False)
print(f"Wrote {UNMATCHED_HOSTS_PATH} ({len(unmatched_hosts):,} distinct hosts)")
print(
    f"Unknown host rows: {int(unknown_host_mask.sum()):,} / {len(out):,} "
    f"({100 * float(unknown_host_mask.mean()):.2f}%)"
)
display(unmatched_hosts.head(40))

Wrote /workspaces/vector-guard/data/02_interim/virus_unmatched_hosts.csv (521 distinct hosts)
Unknown host rows: 1,548 / 168,307 (0.92%)


,host,rows,n_sources,top_source,sample_isolation_source
0,Serum,259,1,ena_sequence,Homo sapiens
1,Apis mellifera,47,1,ena_read_run,honey bee brood comb
2,Denv4,11,1,ena_sequence,serum
3,West Nile virus,11,1,ena_read_run,DNA:ss Viral cDNA
4,Apodemus uralensis,10,1,bvbrc,liver
5,Barbarine ewe,10,1,ena_sequence,blood
6,Bison bison,10,1,ena_sequence,
7,C.pulicaris,10,1,ena_sequence,C.pulicaris
8,Hemophagous bugs,10,1,ena_sequence,
9,Hippobosca equina,10,1,ena_sequence,insect homogenate


## Assert + write parquet

In [7]:
required_cols = [
    "record_id",
    "source",
    "pathogen_name",
    "vector_group",
    "country_iso2",
    "subnational_region",
    "nuts_id",
    "lat",
    "lon",
    "coord_source",
    "geo_precision",
    "h3_cell",
    "year",
    "month",
    "accession",
    "host",
    "host_group",
]
assert list(out.columns) == required_cols
assert out["geo_precision"].isin(GEO_PRECISIONS).all()
assert out["host_group"].isin(HOST_GROUPS).all()
assert "isolation_source" not in out.columns
assert out["record_id"].ne("").all()

# Spot-check locked host_group edge cases on the classified pairs
_edge = [
    ("Homo sapiens", "passage 3 in Aedes albopictus C6/36", "human"),
    ("Aedes aegypti cells", "", "lab"),
    ("", "mosquito pool", "mosquito"),
    ("", "serum", "unknown"),
    ("equid_outbreaks:1;bird_outbreaks:2", "", "bird"),
]
for _host, _iso, _expect in _edge:
    _got = classify_host_group(
        _host,
        host_alias_map=host_alias_map,
        taxon_to_group=host_taxon_lookup,
        isolation_source=_iso,
    )
    assert _got == _expect, (_host, _iso, _got, _expect)

has_coords = out["lat"].notna() & out["lon"].notna()
assert out.loc[has_coords, "lat"].between(-90, 90).all()
assert out.loc[has_coords, "lon"].between(-180, 180).all()
assert (
    out.loc[has_coords, "geo_precision"]
    .isin({GEO_PRECISION_EXACT, GEO_PRECISION_CENTROID})
    .all()
)
assert out.loc[has_coords, "h3_cell"].notna().all()
assert out.loc[~has_coords, "h3_cell"].isna().all()
assert out.loc[~has_coords, "geo_precision"].eq(GEO_PRECISION_COUNTRY).all()
assert (out.loc[out["month"].notna(), "month"].between(1, 12)).all()

# Exact GPS must win over centroid when both would apply
exact_gps_rows = out["coord_source"].isin({"exact_gps", "sample_backfill"}) & has_coords
assert out.loc[exact_gps_rows, "geo_precision"].eq(GEO_PRECISION_EXACT).all()

out.to_parquet(OUT_PATH, index=False)
print(f"Wrote {OUT_PATH} ({len(out):,} rows)")
display(out["geo_precision"].value_counts())
display(out.head(10))
display(out.tail(5))

Wrote /workspaces/vector-guard/data/02_interim/viruses_normalized.parquet (168,307 rows)


geo_precision
country_only            90822
subnational_centroid    41735
exact_coords            35750
Name: count, dtype: int64

,record_id,source,pathogen_name,vector_group,country_iso2,subnational_region,nuts_id,lat,lon,coord_source,geo_precision,h3_cell,year,month,accession,host,host_group
0,ena_sequence:MH170264,ena_sequence,West Nile virus,mosquito,US,"GA, Muscogee County",,32.8547,-83.4078,,subnational_centroid,8444c01ffffffff,<NA>,<NA>,MH170264,Baeolophus bicolor,bird
1,ena_sequence:MH819470,ena_sequence,West Nile virus,mosquito,CA,,,NaN,NaN,,country_only,NaN,2014,9,MH819470,Culex sp.,mosquito
2,ena_sequence:MW803141,ena_sequence,West Nile virus,mosquito,RS,Zemun,,NaN,NaN,,country_only,NaN,<NA>,<NA>,MW803141,Culex pipiens,mosquito
3,ena_sequence:MW835355,ena_sequence,West Nile virus,mosquito,IT,,,NaN,NaN,,country_only,NaN,<NA>,<NA>,MW835355,magpie,bird
4,ena_sequence:MW835358,ena_sequence,West Nile virus,mosquito,IT,,,NaN,NaN,,country_only,NaN,<NA>,<NA>,MW835358,carrion crow,bird
5,ena_sequence:MW862090,ena_sequence,West Nile virus,mosquito,IT,,,NaN,NaN,,country_only,NaN,<NA>,<NA>,MW862090,swift,unknown
6,ena_sequence:MW862097,ena_sequence,West Nile virus,mosquito,IT,,,NaN,NaN,,country_only,NaN,<NA>,<NA>,MW862097,goshawk,bird
7,ena_sequence:MW862099,ena_sequence,West Nile virus,mosquito,IT,,,NaN,NaN,,country_only,NaN,<NA>,<NA>,MW862099,hooded crow,bird
8,ena_sequence:MW862102,ena_sequence,West Nile virus,mosquito,IT,,,NaN,NaN,,country_only,NaN,<NA>,<NA>,MW862102,magpie,bird
9,ena_sequence:MW862107,ena_sequence,West Nile virus,mosquito,IT,,,NaN,NaN,,country_only,NaN,<NA>,<NA>,MW862107,hooded crow,bird


,record_id,source,pathogen_name,vector_group,country_iso2,subnational_region,nuts_id,lat,lon,coord_source,geo_precision,h3_cell,year,month,accession,host,host_group
168302,ecdc_atlas:monthly:ES533:2026-W45,ecdc_atlas,West Nile virus,mosquito,ES,Menorca,ES533,39.950076,4.129265,,subnational_centroid,8439433ffffffff,2026,11,,equid_outbreaks:1,livestock
168303,ecdc_atlas:monthly:ES611:2026-W45,ecdc_atlas,West Nile virus,mosquito,ES,Almería,ES611,37.210875,-2.312152,,subnational_centroid,8438243ffffffff,2026,11,,equid_outbreaks:1,livestock
168304,ecdc_atlas:monthly:ES616:2026-W45,ecdc_atlas,West Nile virus,mosquito,ES,Jaén,ES616,38.017816,-3.444921,,subnational_centroid,84390a1ffffffff,2026,11,,equid_outbreaks:1;bird_outbreaks:1,bird
168305,ecdc_atlas:monthly:ES705:2026-W45,ecdc_atlas,West Nile virus,mosquito,ES,Gran Canaria,ES705,27.958743,-15.585429,,subnational_centroid,84344cbffffffff,2026,11,,equid_outbreaks:3,livestock
168306,ecdc_atlas:monthly:CH033:2026-W45,ecdc_atlas,West Nile virus,mosquito,CH,Aargau,CH033,47.411178,8.154324,,subnational_centroid,841f83bffffffff,2026,11,,equid_outbreaks:1,livestock
